脸部细节识别

In [ ]:
# mediapipe  ## pip install mediapipe
# 468个人脸关键点检测+追踪

import cv2
import mediapipe as mp
import numpy as np
from pathlib import Path
from tqdm import tqdm

# 安装：pip install mediapipe

def extract_face_mediapipe(image_path, output_path, margin=0.3):
    """
    使用MediaPipe检测人脸并抠图
    
    Args:
        margin: 扩展边距比例（0.3表示扩展30%）
    """
    # 初始化
    mp_face_detection = mp.solutions.face_detection
    face_detection = mp_face_detection.FaceDetection(
        model_selection=1,  # 0: 2米内, 1: 5米内
        min_detection_confidence=0.5
    )
    
    # 读取图片
    image = cv2.imread(str(image_path))
    if image is None:
        return None
    
    h, w = image.shape[:2]
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # 检测人脸
    results = face_detection.process(rgb_image)
    
    if not results.detections:
        return None
    
    # 获取第一个人脸的bbox
    detection = results.detections[0]
    bbox = detection.location_data.relative_bounding_box
    
    # 转换为像素坐标并扩展边距
    x = int(bbox.xmin * w)
    y = int(bbox.ymin * h)
    fw = int(bbox.width * w)
    fh = int(bbox.height * h)
    
    # 扩展边距
    margin_w = int(fw * margin)
    margin_h = int(fh * margin)
    
    x = max(0, x - margin_w)
    y = max(0, y - margin_h)
    fw = min(w - x, fw + 2 * margin_w)
    fh = min(h - y, fh + 2 * margin_h)
    
    # 裁剪人脸
    face = image[y:y+fh, x:x+fw]
    
    # 保存
    cv2.imwrite(str(output_path), face)
    
    return face

def batch_extract_faces(input_dir, output_dir, margin=0.3):
    """批量抠人脸"""
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # 获取图片
    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        image_files.extend(input_path.glob(ext))
    
    print(f"Found {len(image_files)} images")
    
    success = 0
    failed = 0
    
    for img_file in tqdm(image_files, desc="Extracting faces"):
        output_file = output_path / img_file.name
        result = extract_face_mediapipe(img_file, output_file, margin)
        
        if result is not None:
            success += 1
        else:
            failed += 1
    
    print(f"\n✅ Success: {success}")
    print(f"❌ Failed (no face detected): {failed}")

# 使用
if __name__ == '__main__':
    batch_extract_faces(
        input_dir='path/to/images',
        output_dir='path/to/faces',
        margin=0.3  # 扩展30%边距
    )

In [ ]:
# Face-Parsing（最精确的人脸抠图）  ## pip install face-alignment

import cv2
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm

# 安装：pip install face-alignment

def extract_face_parsing(image_path, output_path, include_hair=True):
    """
    使用face-parsing精确抠出人脸
    需要先下载模型：https://github.com/zllrunning/face-parsing.PyTorch
    """
    from model import BiSeNet
    
    # 加载模型（首次运行需要下载）
    model = BiSeNet(n_classes=19)
    model.load_state_dict(torch.load('79999_iter.pth'))
    model.eval()
    model.cuda() if torch.cuda.is_available() else model.cpu()
    
    # 读取图片
    image = cv2.imread(str(image_path))
    if image is None:
        return None
    
    h, w = image.shape[:2]
    
    # 预处理
    img_input = cv2.resize(image, (512, 512))
    img_input = img_input / 255.0
    img_input = torch.from_numpy(img_input.transpose(2, 0, 1)).float()
    img_input = img_input.unsqueeze(0)
    
    if torch.cuda.is_available():
        img_input = img_input.cuda()
    
    # 推理
    with torch.no_grad():
        out = model(img_input)[0]
        parsing = out.squeeze(0).cpu().numpy().argmax(0)
    
    # Resize回原始尺寸
    parsing = cv2.resize(parsing.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
    
    # 创建人脸mask
    # 标签：1-skin, 2-l_brow, 3-r_brow, 4-l_eye, 5-r_eye, 
    #      6-eye_g, 7-l_ear, 8-r_ear, 9-nose, 10-mouth, 11-u_lip, 
    #      12-l_lip, 13-neck, 17-hair
    face_labels = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
    if include_hair:
        face_labels.append(17)
    
    mask = np.isin(parsing, face_labels).astype(np.uint8) * 255
    
    # 应用mask
    result = cv2.bitwise_and(image, image, mask=mask)
    
    # 获取bbox并裁剪
    coords = cv2.findNonZero(mask)
    if coords is not None:
        x, y, fw, fh = cv2.boundingRect(coords)
        result = result[y:y+fh, x:x+fw]
    
    # 保存
    cv2.imwrite(str(output_path), result)
    
    return result

In [ ]:
# rembg（通用背景移除，简单粗暴）  ## pip install rembg

from rembg import remove
from PIL import Image
from pathlib import Path
from tqdm import tqdm

# 安装：pip install rembg

def extract_face_rembg(image_path, output_path):
    """使用rembg移除背景"""
    input_img = Image.open(image_path)
    output_img = remove(input_img)
    output_img.save(output_path)

def batch_rembg(input_dir, output_dir):
    """批量背景移除"""
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        image_files.extend(input_path.glob(ext))
    
    for img_file in tqdm(image_files, desc="Removing background"):
        output_file = output_path / f"{img_file.stem}.png"
        extract_face_rembg(img_file, output_file)

# 使用
if __name__ == '__main__':
    batch_rembg(
        input_dir='path/to/images',
        output_dir='path/to/output'
    )

In [ ]:
# InsightFace（功能强大）  ## pip install insightface onnxruntime

import cv2
import insightface
from insightface.app import FaceAnalysis
from pathlib import Path
from tqdm import tqdm

# 安装：pip install insightface onnxruntime

def extract_face_insightface(image_path, output_path, margin=0.2):
    """使用InsightFace检测并抠人脸"""
    
    # 初始化（首次运行会自动下载模型）
    app = FaceAnalysis(providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    app.prepare(ctx_id=0, det_size=(640, 640))
    
    # 读取图片
    image = cv2.imread(str(image_path))
    if image is None:
        return None
    
    # 检测人脸
    faces = app.get(image)
    
    if len(faces) == 0:
        return None
    
    # 获取第一个人脸
    face = faces[0]
    bbox = face.bbox.astype(int)
    
    # 扩展边距
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    
    margin_w = int(w * margin)
    margin_h = int(h * margin)
    
    x1 = max(0, x1 - margin_w)
    y1 = max(0, y1 - margin_h)
    x2 = min(image.shape[1], x2 + margin_w)
    y2 = min(image.shape[0], y2 + margin_h)
    
    # 裁剪
    face_img = image[y1:y2, x1:x2]
    
    # 保存
    cv2.imwrite(str(output_path), face_img)
    
    return face_img

def batch_insightface(input_dir, output_dir, margin=0.2):
    """批量抠人脸"""
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        image_files.extend(input_path.glob(ext))
    
    success = 0
    failed = 0
    
    for img_file in tqdm(image_files, desc="Extracting faces"):
        output_file = output_path / img_file.name
        result = extract_face_insightface(img_file, output_file, margin)
        
        if result is not None:
            success += 1
        else:
            failed += 1
    
    print(f"\n✅ Success: {success}")
    print(f"❌ Failed: {failed}")

# 使用
if __name__ == '__main__':
    batch_insightface(
        input_dir='path/to/images',
        output_dir='path/to/faces',
        margin=0.2
    )